# 10 – Mô hình cuối và kiểm chứng

Notebook 09 cho thấy **kết hợp XLS-R + PhoWhisper** tăng mạnh: 61% với một mô hình, 64,6% khi ensemble.
Nhưng PhoWhisper được huấn luyện để **chép lời**, nên có nguy cơ nó nhận ra *tỉnh* qua **địa danh hoặc từ vựng** được nói ra, thay vì qua giọng.
Notebook này kiểm chứng điều đó trước khi đưa con số vào báo cáo:

1. Train lại các mô hình cuối từ đặc trưng đã cache (vài phút), **lưu dự đoán và trọng số** lên Drive.
2. **Kiểm soát nhiễu:** accuracy trên các câu không nhắc địa danh nào; trên các câu mà mô hình chỉ đọc văn bản đoán sai; trên các câu mà LTAS (phổ trung bình) đoán sai.
   So sánh mức giảm của PhoWhisper với XLS-R: nếu PhoWhisper giảm mạnh hơn rõ rệt thì nó đang dựa vào nội dung lời nói.
3. So sánh cặp Geo-LS với LS trên mô hình kết hợp, có khoảng tin cậy 95%.
4. Bộ hình cuối cho báo cáo.

- **Runtime:** GPU T4 hoặc L4, khoảng 10–15 phút. Cần đã chạy notebook 02 và 09.
- **Kết quả:** `results/final/`. Bạn chỉ cần báo Claude là đã chạy xong, Claude sẽ tự đọc từ Drive.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL, TEXT_COL = "province_code", "text"
FEAT_B = "full13L_wav2vec2-xls-r-300m_12s"      # đặc trưng XLS-R toàn câu (notebook 09)
FEAT_C = "enc_PhoWhisper-large"                  # đặc trưng PhoWhisper (notebook 09)
SEEDS = [0, 1, 2]

In [ ]:
import os, io, glob, json, time, warnings
import numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, top_k_accuracy_score
warnings.filterwarnings("ignore")
T0 = time.time()
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
LOCAL = "/content" if os.path.isdir("/content") else os.path.abspath("_work")
LOCAL_DATA, FEAT_DIR, RES_DIR = f"{LOCAL}/vimd16k", f"{DRIVE_ROOT}/features", f"{DRIVE_ROOT}/results"
os.makedirs(LOCAL_DATA, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else None
autocast = lambda: torch.autocast(device_type=device.type, dtype=AMP or torch.bfloat16, enabled=AMP is not None)
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "", "| AMP:", AMP)

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
pidx = prov.set_index(PROVINCE_COL)
NAMES = [pidx.loc[c, "province_name"] for c in CLASSES]
region_of = np.array([pidx.loc[c, "region"] for c in CLASSES])
yt = y[te]; N = len(meta)
GROUPS = {s: (g.index.values, g["row"].values) for s, g in meta.groupby("shard")}
print(f"{C} tỉnh | train {tr.sum()} | valid {va.sum()} | test {te.sum()} | độ dài trung vị {meta['duration'].median():.1f}s, "
      f"tỉ lệ câu dài hơn 12s: {(meta['duration'] > 12).mean():.0%}")

COORD = {
 "CaoBang": (22.666, 106.258), "LangSon": (21.853, 106.761), "QuangNinh": (20.951, 107.080), "HaiPhong": (20.845, 106.688),
 "ThaiBinh": (20.446, 106.336), "NamDinh": (20.420, 106.168), "PhuTho": (21.322, 105.402), "ThaiNguyen": (21.594, 105.848),
 "YenBai": (21.705, 104.875), "TuyenQuang": (21.823, 105.214), "HaGiang": (22.823, 104.984), "LaoCai": (22.486, 103.971),
 "LaiChau": (22.396, 103.458), "SonLa": (21.326, 103.917), "DienBien": (21.386, 103.023), "HoaBinh": (20.817, 105.338),
 "HaNoi": (21.028, 105.854), "HaiDuong": (20.940, 106.333), "NinhBinh": (20.251, 105.975), "VinhPhuc": (21.310, 105.597),
 "HungYen": (20.646, 106.051), "HaNam": (20.541, 105.914), "BacKan": (22.147, 105.835), "BacGiang": (21.273, 106.195),
 "BacNinh": (21.186, 106.076), "ThanhHoa": (19.807, 105.776), "NgheAn": (18.679, 105.682), "HaTinh": (18.343, 105.906),
 "QuangBinh": (17.468, 106.622), "QuangTri": (16.816, 107.100), "ThuaThienHue": (16.464, 107.586), "DaNang": (16.054, 108.202),
 "QuangNam": (15.573, 108.474), "QuangNgai": (15.120, 108.792), "BinhDinh": (13.776, 109.224), "PhuYen": (13.096, 109.309),
 "KhanhHoa": (12.239, 109.197), "NinhThuan": (11.565, 108.988), "BinhThuan": (10.928, 108.102), "KonTum": (14.350, 108.000),
 "GiaLai": (13.983, 108.000), "DakLak": (12.667, 108.038), "DakNong": (12.004, 107.690), "LamDong": (11.940, 108.458),
 "HoChiMinh": (10.776, 106.701), "DongNai": (10.957, 106.843), "BinhDuong": (10.980, 106.651), "BaRiaVungTau": (10.496, 107.169),
 "TayNinh": (11.310, 106.098), "BinhPhuoc": (11.535, 106.883), "LongAn": (10.535, 106.413), "TienGiang": (10.360, 106.360),
 "BenTre": (10.241, 106.376), "VinhLong": (10.254, 105.972), "TraVinh": (9.935, 106.345), "CanTho": (10.045, 105.747),
 "HauGiang": (9.784, 105.470), "SocTrang": (9.603, 105.980), "BacLieu": (9.294, 105.727), "CaMau": (9.176, 105.150),
 "DongThap": (10.460, 105.633), "AnGiang": (10.386, 105.435), "KienGiang": (10.012, 105.081),
}
LAT = np.array([COORD[n][0] for n in NAMES]); LON = np.array([COORD[n][1] for n in NAMES])
p_ = np.pi / 180
DIST = 2 * 6371 * np.arcsin(np.sqrt(np.sin((LAT[None] - LAT[:, None]) * p_ / 2) ** 2 + np.cos(LAT[:, None] * p_)
                                    * np.cos(LAT[None] * p_) * np.sin((LON[None] - LON[:, None]) * p_ / 2) ** 2))
EYE = torch.eye(C, device=device); DIST_t = torch.tensor(DIST, dtype=torch.float32, device=device)

def target_matrix(kind, p):
    if kind == "ls":
        return (1 - p) * EYE + p / C
    if kind == "geols":
        eps, tau = p
        nb = torch.exp(-DIST_t / tau) * (1 - EYE)
        return (1 - eps) * EYE + eps * nb / nb.sum(1, keepdim=True)
    return EYE

def metrics(P, yy):
    pred = P.argmax(1); d = DIST[yy, pred]
    return dict(acc=(pred == yy).mean(), macro_f1=f1_score(yy, pred, average="macro"),
                top5=top_k_accuracy_score(yy, P, k=min(5, C - 1), labels=np.arange(C)),
                region_acc=(region_of[pred] == region_of[yy]).mean(), mean_err_km=d.mean(),
                within_100km=(d <= 100).mean())

rep = f"{RES_DIR}/geo_ls_report_13layers.json"
ch = json.load(open(rep if os.path.exists(rep) else f"{RES_DIR}/geo_ls_report.json"))["chosen"]
LOSSES = {"ls": ("ls", float(ch["ls"])), "geols": ("geols", tuple(float(v) for v in ch["geols"].strip("()").split(",")))}
print("Hàm mất mát:", LOSSES)

OUT = f"{RES_DIR}/final"; os.makedirs(OUT, exist_ok=True)

def load_feat(name):
    first = np.load(f"{FEAT_DIR}/{name}/{next(iter(GROUPS))}.npy")
    X = np.zeros((N,) + first.shape[1:], np.float16)
    for s, (idx, _) in GROUPS.items():
        X[idx] = np.load(f"{FEAT_DIR}/{name}/{s}.npy")
    Xg = torch.tensor(X, device=device); mtr = torch.tensor(tr, device=device)
    return Xg, Xg[mtr].float().mean(0), Xg[mtr].float().std(0) + 1e-5
FEATS = {"B": load_feat(FEAT_B), "C": load_feat(FEAT_C)}
print({k: tuple(v[0].shape) for k, v in FEATS.items()})

## 1. Train lại các mô hình cuối (giống notebook 09) và lưu dự đoán, trọng số

In [ ]:
class Head(nn.Module):
    def __init__(s, keys):
        super().__init__(); s.keys = keys
        for k in keys:
            _, mu, sd = FEATS[k]
            s.register_buffer(f"mu_{k}", mu); s.register_buffer(f"sd_{k}", sd)
        s.w = nn.ParameterDict({k: nn.Parameter(torch.zeros(FEATS[k][1].shape[0])) for k in keys})
        s.mlp = nn.Sequential(nn.Linear(sum(FEATS[k][1].shape[1] for k in keys), 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
    def forward(s, idx):
        parts = []
        for k in s.keys:
            z = (FEATS[k][0][idx].float() - getattr(s, f"mu_{k}")) / getattr(s, f"sd_{k}")
            parts.append((z * s.w[k].softmax(0)[None, :, None]).sum(1))
        return s.mlp(torch.cat(parts, -1))

@torch.no_grad()
def predict(h, mask, bs=2048):
    h.eval(); idx = torch.tensor(np.where(mask)[0], device=device)
    return torch.cat([h(idx[i:i + bs]).softmax(-1) for i in range(0, len(idx), bs)]).cpu().numpy()

def train_head(keys, loss, seed, epochs=80, lr=1e-3, wd=1e-3, bs=256, patience=10):
    torch.manual_seed(seed)
    h = Head(keys).to(device); opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=wd)
    Q = target_matrix(*loss); tri = torch.tensor(np.where(tr)[0], device=device); Y = torch.tensor(y, device=device)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        h.train(); perm = tri[torch.randperm(len(tri), device=device)]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]; l = -(Q[Y[b]] * F.log_softmax(h(b), -1)).sum(1).mean()
            opt.zero_grad(); l.backward(); opt.step()
        acc = (predict(h, va).argmax(1) == y[va]).mean()
        if acc > best:
            best, state, bad = acc, {k: v.clone() for k, v in h.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    h.load_state_dict(state)
    return h

SETS = {"B": ["B"], "C": ["C"], "D": ["B", "C"]}
PV, PT = {}, {}
for sname, keys in SETS.items():
    for lname, loss in LOSSES.items():
        for s in SEEDS:
            h = train_head(keys, loss, s)
            PV[(sname, lname, s)], PT[(sname, lname, s)] = predict(h, va), predict(h, te)
            if sname == "D":
                torch.save({k: v.cpu() for k, v in h.state_dict().items()}, f"{OUT}/head_D_{lname}_seed{s}.pt")
        print(f"{sname} {lname}: test acc (từng seed) {[round(float((PT[(sname, lname, s)].argmax(1) == yt).mean()), 4) for s in SEEDS]}", flush=True)

ens = lambda keys: (np.mean([PV[k] for k in keys], 0), np.mean([PT[k] for k in keys], 0))
MODELS = {
    "B_xlsr|ls": [("B", "ls", s) for s in SEEDS],
    "C_phowhisper|ls": [("C", "ls", s) for s in SEEDS],
    "D_fusion|ls": [("D", "ls", s) for s in SEEDS],
    "D_fusion|geols": [("D", "geols", s) for s in SEEDS],
    "FINAL_acc: B+C+D|ls": [(sn, "ls", s) for sn in SETS for s in SEEDS],
}
P = {m: ens(k) for m, k in MODELS.items()}
for m, (pv, pt) in P.items():
    np.save(f"{OUT}/test_probs_{m.split(':')[0].replace('|', '_').replace('+', '')}.npy", pt)

B_IDX = np.random.default_rng(0).integers(0, len(yt), (2000, len(yt)))
def ci(vals):
    return np.percentile(vals, [2.5, 97.5]).round(4).tolist()
def boot_metrics(pred):
    ok = (pred == yt).astype(float)
    f1s = [f1_score(yt[b], pred[b], average="macro") for b in B_IDX[:500]]
    return dict(acc_ci95=ci(ok[B_IDX].mean(1)), macro_f1_ci95=ci(f1s))
TABLE = []
for m, (pv, pt) in P.items():
    TABLE.append(dict(model=m, **{f"valid_{k}": round(float(v), 4) for k, v in metrics(pv, y[va]).items()},
                      **{f"test_{k}": round(float(v), 4) for k, v in metrics(pt, yt).items()}, **boot_metrics(pt.argmax(1))))
TABLE = pd.DataFrame(TABLE)
print(TABLE[["model", "valid_acc", "test_acc", "acc_ci95", "test_macro_f1", "macro_f1_ci95", "test_region_acc", "test_mean_err_km"]].to_string(index=False))

def paired(a, b):
    dacc = (a == yt).astype(float) - (b == yt); dkm = DIST[yt, a] - DIST[yt, b]
    return dict(acc_diff=round(float(dacc.mean()), 4), acc_ci95=ci(dacc[B_IDX].mean(1)),
                km_diff=round(float(dkm.mean()), 1), km_ci95=np.percentile(dkm[B_IDX].mean(1), [2.5, 97.5]).round(1).tolist())
pred = {m: v[1].argmax(1) for m, v in P.items()}
pred["baseline_xlsr25_ce (02)"] = np.load(f"{RES_DIR}/pred_xlsr_wsum_mlp.npy")
COMPARE = {
    "D_fusion|geols - D_fusion|ls": paired(pred["D_fusion|geols"], pred["D_fusion|ls"]),
    "D_fusion|ls - C_phowhisper|ls": paired(pred["D_fusion|ls"], pred["C_phowhisper|ls"]),
    "D_fusion|ls - B_xlsr|ls": paired(pred["D_fusion|ls"], pred["B_xlsr|ls"]),
    "FINAL_acc - baseline_xlsr25_ce (02)": paired(pred["FINAL_acc: B+C+D|ls"], pred["baseline_xlsr25_ce (02)"]),
    "D_fusion|geols - baseline_xlsr25_ce (02)": paired(pred["D_fusion|geols"], pred["baseline_xlsr25_ce (02)"]),
}
print(json.dumps(COMPARE, indent=1))

## 2. Kiểm soát nhiễu: mô hình mới có dựa vào địa danh, từ vựng hay kênh thu không?

In [ ]:
import re, unicodedata
GEO = {  # key: tên trong provinces.csv -> (tên tiếng Việt, vĩ độ, kinh độ, [địa danh phụ])
 "CaoBang": ("Cao Bằng", 22.666, 106.258, []), "LangSon": ("Lạng Sơn", 21.853, 106.761, []),
 "QuangNinh": ("Quảng Ninh", 20.951, 107.080, ["Hạ Long", "Cẩm Phả", "Móng Cái"]),
 "HaiPhong": ("Hải Phòng", 20.845, 106.688, []), "ThaiBinh": ("Thái Bình", 20.446, 106.336, []),
 "NamDinh": ("Nam Định", 20.420, 106.168, []), "PhuTho": ("Phú Thọ", 21.322, 105.402, ["Việt Trì"]),
 "ThaiNguyen": ("Thái Nguyên", 21.594, 105.848, []), "YenBai": ("Yên Bái", 21.705, 104.875, []),
 "TuyenQuang": ("Tuyên Quang", 21.823, 105.214, []), "HaGiang": ("Hà Giang", 22.823, 104.984, []),
 "LaoCai": ("Lào Cai", 22.486, 103.971, ["Sa Pa"]), "LaiChau": ("Lai Châu", 22.396, 103.458, []),
 "SonLa": ("Sơn La", 21.326, 103.917, []), "DienBien": ("Điện Biên", 21.386, 103.023, []),
 "HoaBinh": ("Hòa Bình", 20.817, 105.338, []), "HaNoi": ("Hà Nội", 21.028, 105.854, []),
 "HaiDuong": ("Hải Dương", 20.940, 106.333, []), "NinhBinh": ("Ninh Bình", 20.251, 105.975, []),
 "VinhPhuc": ("Vĩnh Phúc", 21.310, 105.597, ["Vĩnh Yên"]), "HungYen": ("Hưng Yên", 20.646, 106.051, []),
 "HaNam": ("Hà Nam", 20.541, 105.914, ["Phủ Lý"]), "BacKan": ("Bắc Kạn", 22.147, 105.835, ["Bắc Cạn"]),
 "BacGiang": ("Bắc Giang", 21.273, 106.195, []), "BacNinh": ("Bắc Ninh", 21.186, 106.076, []),
 "ThanhHoa": ("Thanh Hóa", 19.807, 105.776, []), "NgheAn": ("Nghệ An", 18.679, 105.682, []),
 "HaTinh": ("Hà Tĩnh", 18.343, 105.906, []), "QuangBinh": ("Quảng Bình", 17.468, 106.622, ["Đồng Hới"]),
 "QuangTri": ("Quảng Trị", 16.816, 107.100, ["Đông Hà"]), "ThuaThienHue": ("Thừa Thiên Huế", 16.464, 107.586, ["Huế"]),
 "DaNang": ("Đà Nẵng", 16.054, 108.202, []), "QuangNam": ("Quảng Nam", 15.573, 108.474, ["Tam Kỳ", "Hội An"]),
 "QuangNgai": ("Quảng Ngãi", 15.120, 108.792, []), "BinhDinh": ("Bình Định", 13.776, 109.224, ["Quy Nhơn"]),
 "PhuYen": ("Phú Yên", 13.096, 109.309, ["Tuy Hòa"]), "KhanhHoa": ("Khánh Hòa", 12.239, 109.197, ["Nha Trang", "Cam Ranh"]),
 "NinhThuan": ("Ninh Thuận", 11.565, 108.988, ["Phan Rang"]), "BinhThuan": ("Bình Thuận", 10.928, 108.102, ["Phan Thiết"]),
 "KonTum": ("Kon Tum", 14.350, 108.000, []), "GiaLai": ("Gia Lai", 13.983, 108.000, ["Pleiku", "Plây Cu"]),
 "DakLak": ("Đắk Lắk", 12.667, 108.038, ["Buôn Ma Thuột", "Đắc Lắc"]), "DakNong": ("Đắk Nông", 12.004, 107.690, ["Gia Nghĩa"]),
 "LamDong": ("Lâm Đồng", 11.940, 108.458, ["Đà Lạt"]), "HoChiMinh": ("Hồ Chí Minh", 10.776, 106.701, ["Sài Gòn"]),
 "DongNai": ("Đồng Nai", 10.957, 106.843, ["Biên Hòa"]), "BinhDuong": ("Bình Dương", 10.980, 106.651, ["Thủ Dầu Một"]),
 "BaRiaVungTau": ("Bà Rịa", 10.496, 107.169, ["Vũng Tàu"]), "TayNinh": ("Tây Ninh", 11.310, 106.098, []),
 "BinhPhuoc": ("Bình Phước", 11.535, 106.883, ["Đồng Xoài"]), "LongAn": ("Long An", 10.535, 106.413, []),
 "TienGiang": ("Tiền Giang", 10.360, 106.360, ["Mỹ Tho"]), "BenTre": ("Bến Tre", 10.241, 106.376, []),
 "VinhLong": ("Vĩnh Long", 10.254, 105.972, []), "TraVinh": ("Trà Vinh", 9.935, 106.345, []),
 "CanTho": ("Cần Thơ", 10.045, 105.747, []), "HauGiang": ("Hậu Giang", 9.784, 105.470, ["Vị Thanh"]),
 "SocTrang": ("Sóc Trăng", 9.603, 105.980, []), "BacLieu": ("Bạc Liêu", 9.294, 105.727, []),
 "CaMau": ("Cà Mau", 9.176, 105.150, []), "DongThap": ("Đồng Tháp", 10.460, 105.633, ["Cao Lãnh", "Sa Đéc"]),
 "AnGiang": ("An Giang", 10.386, 105.435, ["Long Xuyên", "Châu Đốc"]), "KienGiang": ("Kiên Giang", 10.012, 105.081, ["Rạch Giá", "Phú Quốc"]),
}


TONE_FIX = {"oà": "òa", "oá": "óa", "oả": "ỏa", "oã": "õa", "oạ": "ọa", "oè": "òe", "oé": "óe", "oẻ": "ỏe",
            "oẽ": "õe", "oẹ": "ọe", "uỳ": "ùy", "uý": "úy", "uỷ": "ủy", "uỹ": "ũy", "uỵ": "ụy"}

def norm(s):
    s = unicodedata.normalize("NFC", str(s).lower())
    for a, b in TONE_FIX.items():
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s)

place_terms = {k: [norm(GEO[n][0])] + [norm(a) for a in GEO[n][3]] for k, n in enumerate(NAMES)}
all_terms = sorted({t for ts in place_terms.values() for t in ts}, key=len, reverse=True)
PLACE_RE = re.compile(r"(?<!\w)(" + "|".join(map(re.escape, all_terms)) + r")(?!\w)")
own_re = {k: re.compile(r"(?<!\w)(" + "|".join(map(re.escape, ts)) + r")(?!\w)") for k, ts in place_terms.items()}


texts = np.array([norm(t) for t in meta[TEXT_COL].fillna("")])
mentions_any = np.array([bool(PLACE_RE.search(t)) for t in texts])[te]
mentions_own = np.array([bool(own_re[k].search(t)) for k, t in zip(y, texts)])[te]
p_text, p_ltas = np.load(f"{RES_DIR}/pred_text_tfidf.npy"), np.load(f"{RES_DIR}/pred_acoustic_ltas.npy")
SUBSETS = {"all": np.ones(len(yt), bool), "no_place_name": ~mentions_any, "mentions_own_province": mentions_own,
           "text_wrong": p_text != yt, "ltas_wrong": p_ltas != yt, "text_and_ltas_wrong": (p_text != yt) & (p_ltas != yt)}
rows = []
for sn, m in SUBSETS.items():
    r = dict(subset=sn, n=int(m.sum()))
    for k in ["baseline_xlsr25_ce (02)", "B_xlsr|ls", "C_phowhisper|ls", "D_fusion|ls", "D_fusion|geols", "FINAL_acc: B+C+D|ls"]:
        ok = (pred[k][m] == yt[m]).astype(float)
        r[k] = round(float(ok.mean()), 4)
        if k in ("C_phowhisper|ls", "D_fusion|ls"):
            r[k + " ci95"] = ci(ok[np.random.default_rng(1).integers(0, len(ok), (1000, len(ok)))].mean(1)) if len(ok) else None
    rows.append(r)
CONF = pd.DataFrame(rows)
print(CONF.to_string(index=False))
base = CONF.set_index("subset")
RETAIN = {k: {sn: round(float(base.loc[sn, k] / base.loc["all", k]), 3) for sn in ("no_place_name", "text_wrong", "text_and_ltas_wrong")}
          for k in ["baseline_xlsr25_ce (02)", "B_xlsr|ls", "C_phowhisper|ls", "D_fusion|ls"]}
print("Tỉ lệ giữ lại accuracy (tập con / toàn bộ):", json.dumps(RETAIN, ensure_ascii=False, indent=1))

## 3. Hình cho báo cáo

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                     "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e"})
FIN = "D_fusion|geols"; pf = pred[FIN]
acc_prov = np.array([(pf[yt == k] == k).mean() for k in range(C)])
REG = {"North": ("#2a78d6", "o", "Bắc"), "Central": ("#eb6834", "s", "Trung"), "South": ("#1baf7a", "^", "Nam")}
fig, ax = plt.subplots(figsize=(4.2, 6.4))
for rn, (col, mk, lab) in REG.items():
    m = region_of == rn
    ax.scatter(LON[m], LAT[m], s=20 + 260 * acc_prov[m], c=col, marker=mk, alpha=0.85, edgecolors="#fcfcfb", linewidths=2)
for k in np.argsort(acc_prov)[:5]:
    ax.annotate(f"{NAMES[k]} {acc_prov[k]:.0%}", (LON[k], LAT[k]), fontsize=7, color="#52514e", xytext=(4, -3), textcoords="offset points")
ax.set_xlabel("Kinh độ"); ax.set_ylabel("Vĩ độ"); ax.set_aspect("equal")
ax.set_title("Accuracy từng tỉnh – mô hình cuối (D + Geo-LS)", fontsize=9, loc="left")
ax.legend(handles=[Line2D([], [], ls="", marker=mk, color=c, markersize=7, label=l) for c, mk, l in REG.values()],
          frameon=False, loc="lower left", title="Vùng")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_final_province_map.png"); plt.show()

fig, ax = plt.subplots(figsize=(5.4, 3.4))
for k, col, lab in [("baseline_xlsr25_ce (02)", "#52514e", "Baseline: XLS-R 25 tầng, CE"), ("FINAL_acc: B+C+D|ls", "#2a78d6", "Ensemble B+C+D, LS"),
                    ("D_fusion|geols", "#eb6834", "Kết hợp D, Geo-LS")]:
    d = np.sort(DIST[yt, pred[k]]); ax.plot(d, np.arange(1, len(d) + 1) / len(d), color=col, linewidth=2, label=lab)
ax.set_xlabel("Khoảng cách sai (km, 0 = đúng tỉnh)"); ax.set_ylabel("Tỉ lệ tích lũy câu test")
ax.set_title("Phân phối khoảng cách sai", fontsize=9, loc="left"); ax.grid(color="#e6e5e0", linewidth=0.6)
ax.legend(frameon=False, loc="lower right"); fig.tight_layout(); fig.savefig(f"{OUT}/fig_final_error_cdf.png"); plt.show()

M = np.zeros((C, C)); np.add.at(M, (yt, pf), 1); off = ~np.eye(C, dtype=bool)
rho, pval = spearmanr(M[off], DIST[off])

report = dict(table=TABLE.to_dict(orient="records"), compare=COMPARE, confound=CONF.to_dict(orient="records"),
              retain_ratio=RETAIN, per_province_acc_final={NAMES[k]: round(float(a), 4) for k, a in enumerate(acc_prov)},
              confusion_vs_distance=dict(spearman=round(float(rho), 4), p=float(pval)),
              losses={k: str(v[1]) for k, v in LOSSES.items()}, feats=dict(B=FEAT_B, C=FEAT_C),
              gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", total_minutes=round((time.time() - T0) / 60, 1))
json.dump(report, open(f"{OUT}/final_report.json", "w"), ensure_ascii=False, indent=1, default=str)
print("Đã lưu results/final/ – báo Claude là xong.")